# Intro Assignment Notebook

## Part 1

### 1)

In [ ]:
import pandas as pd

df = pd.read_csv("SCYA/SCYA69_AllRVs.csv", encoding="utf-8")

In [ ]:
df.head()

In [ ]:
# Create a new column "ra_wrapped" that is a copy of the "RA" column
df["ra_wrapped"] = df["RA"].copy()
# df["ra_wrapped"] > 180 creates boolean mask of True or False
mask = df["ra_wrapped"] > 180
# loc lets you target a specific subset of the dataframe:
# loc just selects a rows and a columns so df.loc[which_rows, which_columns]
# Then df.loc[mask, "ra_wrapped"] selects the rows where the mask is True and column is "ra_wrapped"
df.loc[mask, "ra_wrapped"] = df.loc[mask, "ra_wrapped"] - 360

In [ ]:
print(df[["RA", "ra_wrapped"]])

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(df["ra_wrapped"], df["Dec"], s = 5)
plt.xlabel("Right Ascension (degrees)")
plt.ylabel("Declination (degrees)")
plt.title("Sky Positions of Stars in SCYA69")
plt.grid()
plt.savefig("figures/sky_positions_SCYA69.png", dpi=300, bbox_inches="tight")
plt.show() 

In [ ]:
plt.scatter(df["pmra"], df["pmdec"], s = 5)
plt.xlabel("Proper Motion in Right Ascension (mas/yr)")
plt.ylabel("Proper Motion in Declination (mas/yr)")
plt.title("Proper Motions of Stars in SCYA69")
plt.grid()
plt.savefig("figures/proper_motions_SCYA69.png", dpi=300, bbox_inches="tight")
plt.show()

### 2)

In [ ]:
###useful functions############################
###note: let me know if any important statements are missing, I may not have gotten them all

import numpy as np
from astropy import units as u
import astropy.coordinates as coord
from astropy.coordinates import SkyCoord

###Imports for pyutils functions:
from pyutils.ACRastro.gal_uvw import gal_uvw
from pyutils.gal_xyz import gal_xyz

def gal(r, d): #input RA/Dec sky coords, output l/b galactic sky coords
    d = SkyCoord(ra=r*u.degree, dec=d*u.degree, frame='icrs')
    gd = d.transform_to('galactic')
    agl, agb = gd.l, gd.b
    return agl.value, agb.value

def pmgal(ra, dec, pmra, pmdec): #input RA, Dec and proper motion in RA, Dec, return proper motion l,b
    ipmt = coord.SkyCoord(ra=ra*u.degree, dec=dec*u.degree, pm_ra_cosdec=pmra*u.mas/u.yr,
                          pm_dec=pmdec*u.mas/u.yr, frame='icrs')
    pmg = ipmt.transform_to(coord.Galactic)
    pml, pmb = pmg.pm_l_cosb, pmg.pm_b
    return pml, pmb

def propmot(ra, dec, dist, U, V, W): #input RA, Dec, Distance, and UVW cartesian velocity, return RV and proper motion in RA, Dec
    # Ensure inputs are numpy arrays
    ra = np.asarray(ra)
    dec = np.asarray(dec)
    dist = np.asarray(dist)
    U = np.asarray(U)
    V = np.asarray(V)
    W = np.asarray(W)

    # Transformation matrix TM
    TM = np.array([[-0.06699, -0.87276, -0.48354],
                   [0.49273, -0.45035, 0.74458],
                   [-0.86760, -0.18837, 0.46020]])

    # Create the output array
    num_points = ra.size
    sol = np.zeros((3, num_points))

    # Loop over each data point
    for i in range(num_points):
        # Trigonometric terms
        cosd = np.cos(np.deg2rad(dec[i]))
        sind = np.sin(np.deg2rad(dec[i]))
        cosa = np.cos(np.deg2rad(ra[i]))
        sina = np.sin(np.deg2rad(ra[i]))

        # Transformation matrix AM for each data point
        AM = np.array([[cosa * cosd, -sina, -cosa * sind],
                       [sina * cosd, cosa, -sina * sind],
                       [sind,        0,    cosd]])

        # Matrix multiplication
        BM = np.matmul(TM, AM).T

        # Solve for radial velocity and proper motions
        sol[:, i] = np.matmul(BM, np.array([U[i], V[i], W[i]]))

    # Convert proper motions to mas/yr
    plx = 1e3 / dist
    sol[1, :] = sol[1, :] * plx / 4.74047
    sol[2, :] = sol[2, :] * plx / 4.74047

    return sol

def Vt(pml, pmb, d): #insert proper motion in l, b and distance, return transverse velocities in l, b
    d = d*u.pc  #distance in pc
    vtl = d*pml.value/206264807.*pctokm*u.km**-1
    vtb = d*pmb.value/206264807.*pctokm*u.km**-1
    return (vtl/ytos).value, (vtb/ytos).value

In [ ]:
# Converting RA/DEC to l/b galactic coordinates:

l, b = gal(df["RA"].values, df["Dec"].values)
plt.figure(figsize=(15, 5))
plt.scatter(l, b, s = 5)
plt.xlabel("Galactic Longitude (degrees)")
plt.ylabel("Galactic Latitude (degrees)")
plt.title("Galactic Positions of Stars in SCYA69")
plt.savefig("figures/galactic_positions_SCYA69.png", dpi=300, bbox_inches="tight")
plt.grid()

plt.show()

In [ ]:
from pyutils.gal_xyz import gal_xyz


In [ ]:
X, Y, Z = gal_xyz(df["RA"].values, df["Dec"].values, df["d"].values, radec= True, plx = False)
# Input is RA/DEC and (radec = True) and parallax is false since input is distance (plx = False)

In [ ]:
help(gal_xyz)

### 3)

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

os.makedirs("figures", exist_ok=True)

fig = plt.figure(figsize=(10, 8))

gs = gridspec.GridSpec(
    2, 2,
    height_ratios=[3, 2],
    width_ratios=[1, 1.6],
    hspace=0.25,
    wspace=0.20
)

ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[0, 1])
ax2 = fig.add_subplot(gs[1, :])

ax0.scatter(X, Y, s=5)
ax0.set_xlabel("X [pc]")
ax0.set_ylabel("Y [pc]")
ax0.grid()

ax1.scatter(X, Z, s=5)
ax1.set_xlabel("X [pc]")
ax1.set_ylabel("Z [pc]")
ax1.grid()

ax2.scatter(Y, Z, s=5)
ax2.set_xlabel("Y [pc]")
ax2.set_ylabel("Z [pc]")
ax2.grid()

fig.savefig("figures/XYZ_positions_SCYA69.png", dpi=300, bbox_inches="tight")

plt.show()


### 4)

In [ ]:
pctokm = 3.0856775814913673e13  # km per pc
ytos = 365.25 * 24 * 60 * 60    # seconds per year

pml, pmb = pmgal(df["RA"].values, df["Dec"].values, df["pmra"].values, df["pmdec"].values)
Vt_l, Vt_b = Vt(pml, pmb, df["d"].values)

### 5)

In [ ]:
def plot_cmd(data, pmem_cut= None):
    """
    Plot Gaia absolute G magnitude vs Gaia BP-RP colour.

    Parameters
    ----------
    data : pandas.DataFrame
        Dataframe containing g, bp, rp, d, and optionally Pmem.
    pmem_cut : float or None
        If given, only plot stars with Pmem > pmem_cut.
    """

    # Pmem cuts for threshold of membership probabilities
    if pmem_cut is not None:
        mask = data["Pmem"] > pmem_cut
        data = data[mask] # Only include data from the boolean mask 

    # Calculate BP-RP color (G) and absolute G magnitude
    bp_rp = data["bp"] - data["rp"]
    M_G = data["g"] - 5*np.log10(data["d"]) + 5

    plt.figure(figsize=(6, 7))
    plt.scatter(bp_rp, M_G, s=5)

    plt.xlabel("Gaia BP - RP")
    plt.ylabel("Absolute Gaia G Magnitude")
    
    if pmem_cut is None:
        plt.title("CMD of SCYA69")
    else:
        plt.title(f"CMD of SCYA69, Pmem > {pmem_cut}")

    plt.gca().invert_yaxis()
    plt.grid()
    plt.savefig(f"figures/CMD_SCYA69_pmem_cut_{pmem_cut}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
plot_cmd(df, pmem_cut=0.6)

In [ ]:
def plot_cmd_highlight(data, pmem_cut=None):
    if pmem_cut is not None:
        data = data[data["Pmem"] > pmem_cut]

    bp_rp = data["bp"] - data["rp"]
    M_G = data["g"] - 5*np.log10(data["d"]) + 5

    plt.figure(figsize=(6, 7))
    sc = plt.scatter(
        bp_rp,
        M_G,
        c=data["Pmem"],
        s=8,
        cmap="viridis",
        alpha=0.8
    )

    plt.colorbar(sc, label="Pmem")
    plt.xlabel("Gaia BP - RP")
    plt.ylabel("Absolute Gaia G")
    plt.title("CMD of SCYA69 colored by membership probability")
    plt.gca().invert_yaxis()
    plt.grid()
    plt.savefig(f"figures/CMD_SCYA69_pmem_colored.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
plot_cmd_highlight(df)

In [ ]:
def continuous_pyoung(df):
    bp_rp = df["bp"] - df["rp"]
    M_G = df["g"] - 5*np.log10(df["d"]) + 5

    fig, axs = plt.subplots(3, 1, figsize=(10,20))  # 3 rows, 1 column
    axs[0].scatter(df["ra_wrapped"], df["Dec"], c=df["Pyoung"], s=5, cmap="viridis")
    axs[0].set_xlabel("Right Ascension (degrees)")
    axs[0].set_ylabel("Declination (degrees)")
    axs[0].set_title("Sky Positions of Stars in SCYA69 colored by Pyoung")

    axs[1].scatter(df["pmra"], df["pmdec"], c=df["Pyoung"], s=5, cmap="viridis")
    axs[1].set_xlabel("Proper Motion in Right Ascension (mas/yr)")
    axs[1].set_ylabel("Proper Motion in Declination (mas/yr)")
    axs[1].set_title("Proper Motions of Stars in SCYA69 colored by Pyoung")

    axs[2].scatter(bp_rp, M_G, c=df["Pyoung"], s=5, cmap="viridis")
    axs[2].set_xlabel("Gaia BP - RP")
    axs[2].set_ylabel("Absolute Gaia G")
    axs[2].invert_yaxis()
    axs[2].set_title("CMD of SCYA69 colored by Pyoung")

    plt.colorbar(axs[0].collections[0], label="Pyoung")
    plt.colorbar(axs[1].collections[0], label="Pyoung")
    plt.colorbar(axs[2].collections[0], label="Pyoung")

continuous_pyoung(df)

In [ ]:
def tri_flag_pyoung(df):

    bp_rp = df["bp"] - df["rp"]
    M_G = df["g"] - 5*np.log10(df["d"]) + 5
    
    # Define masks
    young = df["Pyoung"] > 0.2
    old = df["Pyoung"] < 0.001
    mid = (~young) & (~old)

    # Create figure
    fig, axs = plt.subplots(3, 1, figsize=(10, 20))


    # Spatial Distribution


    axs[0].scatter(df["ra_wrapped"][old],
                df["Dec"][old],
                s=5,
                label="Old")

    axs[0].scatter(df["ra_wrapped"][mid],
                df["Dec"][mid],
                s=5,
                label="Intermediate")

    axs[0].scatter(df["ra_wrapped"][young],
                df["Dec"][young],
                s=5,
                label="Young")

    axs[0].set_xlabel("Right Ascension (degrees)")
    axs[0].set_ylabel("Declination (degrees)")
    axs[0].set_title("Sky Positions classified by Pyoung")
    axs[0].legend()

    # Proper Motion

    axs[1].scatter(df["pmra"][old],
                df["pmdec"][old],
                s=5,
                label="Old")

    axs[1].scatter(df["pmra"][mid],
                df["pmdec"][mid],
                s=5,
                label="Intermediate")

    axs[1].scatter(df["pmra"][young],
                df["pmdec"][young],
                s=5,
                label="Young")

    axs[1].set_xlabel("Proper Motion RA (mas/yr)")
    axs[1].set_ylabel("Proper Motion Dec (mas/yr)")
    axs[1].set_title("Proper Motions classified by Pyoung")
    axs[1].legend()


    # Colour Magnitude Distribution


    axs[2].scatter(bp_rp[old],
                M_G[old],
                s=5,
                label="Old")

    axs[2].scatter(bp_rp[mid],
                M_G[mid],
                s=5,
                label="Intermediate")

    axs[2].scatter(bp_rp[young],
                M_G[young],
                s=5,
                label="Young")

    axs[2].set_xlabel("Gaia BP - RP")
    axs[2].set_ylabel("Absolute Gaia G")
    axs[2].invert_yaxis()
    axs[2].set_title("CMD classified by Pyoung")
    axs[2].legend()

    plt.tight_layout()
    plt.show()

tri_flag_pyoung(df)

In [ ]:
def continuous_strength(df):
    bp_rp = df["bp"] - df["rp"]
    M_G = df["g"] - 5*np.log10(df["d"]) + 5

    fig, axs = plt.subplots(3, 1, figsize=(10, 20))  # 3 rows, 1 column
    axs[0].scatter(df["ra_wrapped"], df["Dec"], c=df["strength"], s=5, cmap="viridis")
    axs[0].set_xlabel("Right Ascension (degrees)")
    axs[0].set_ylabel("Declination (degrees)")
    axs[0].set_title("Sky Positions of Stars in SCYA69 colored by Strength")

    axs[1].scatter(df["pmra"], df["pmdec"], c=df["strength"], s=5, cmap="viridis")
    axs[1].set_xlabel("Proper Motion in Right Ascension (mas/yr)")
    axs[1].set_ylabel("Proper Motion in Declination (mas/yr)")
    axs[1].set_title("Proper Motions of Stars in SCYA69 colored by Strength")

    axs[2].scatter(bp_rp, M_G, c=df["strength"], s=5, cmap="viridis")
    axs[2].set_xlabel("Gaia BP - RP")
    axs[2].set_ylabel("Absolute Gaia G")
    axs[2].invert_yaxis()
    axs[2].set_title("CMD of SCYA69 colored by Strength")

    plt.colorbar(axs[0].collections[0], label="Strength")
    plt.colorbar(axs[1].collections[0], label="Strength")
    plt.colorbar(axs[2].collections[0], label="Strength")

continuous_strength(df)

In [ ]:
def binary_founding(df):
    bp_rp = df["bp"] - df["rp"]
    M_G = df["g"] - 5*np.log10(df["d"]) + 5

    fig, axs = plt.subplots(3, 1, figsize=(10, 20))  # 3 rows, 1 column
    axs[0].scatter(df["ra_wrapped"], df["Dec"], c=df["founding"], s=5, cmap='bwr')
    axs[0].set_xlabel("Right Ascension (degrees)")
    axs[0].set_ylabel("Declination (degrees)")
    axs[0].set_title("Sky Positions of Stars in SCYA69 colored by Founding Flag")

    axs[1].scatter(df["pmra"], df["pmdec"], c=df["founding"], s=5, cmap='bwr')
    axs[1].set_xlabel("Proper Motion in Right Ascension (mas/yr)")
    axs[1].set_ylabel("Proper Motion in Declination (mas/yr)")
    axs[1].set_title("Proper Motions of Stars in SCYA69 colored by Founding Flag")

    axs[2].scatter(bp_rp, M_G, c=df["founding"], s=5, cmap='bwr')
    axs[2].set_xlabel("Gaia BP - RP")
    axs[2].set_ylabel("Absolute Gaia G")
    axs[2].invert_yaxis()
    axs[2].set_title("CMD of SCYA69 colored by Founding Flag")

    plt.colorbar(axs[0].collections[0], label="Founding Flag")
    plt.colorbar(axs[1].collections[0], label="Founding Flag")
    plt.colorbar(axs[2].collections[0], label="Founding Flag")

binary_founding(df)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def add_binary_flags(df, pyoung_cutoff=0.2, strength_cutoff=0.5):
    df = df.copy()

    if "ra_wrapped" not in df.columns:
        df["ra_wrapped"] = df["RA"].copy()
        mask = df["ra_wrapped"] > 180
        df.loc[mask, "ra_wrapped"] = df.loc[mask, "ra_wrapped"] - 360

    df["Pyoung_flag"] = df["Pyoung"] >= pyoung_cutoff
    df["strength_flag"] = df["strength"] >= strength_cutoff
    df["founding_flag"] = df["founding"].astype(bool)

    return df


def plot_binary_comparison(df, pyoung_cutoff=0.2, strength_cutoff=0.5):
    df = add_binary_flags(
        df,
        pyoung_cutoff=pyoung_cutoff,
        strength_cutoff=strength_cutoff
    )

    bp_rp = df["bp"] - df["rp"]
    M_G = df["g"] - 5 * np.log10(df["d"]) + 5

    flags = {
        f"Pyoung >= {pyoung_cutoff}": "Pyoung_flag",
        f"Strength >= {strength_cutoff}": "strength_flag",
        "Founding Flag": "founding_flag",
    }

    views = [
        {
            "name": "Spatial",
            "x": df["ra_wrapped"],
            "y": df["Dec"],
            "xlabel": "Right Ascension (degrees)",
            "ylabel": "Declination (degrees)",
            "invert_y": False,
        },
        {
            "name": "Proper Motion",
            "x": df["pmra"],
            "y": df["pmdec"],
            "xlabel": "Proper Motion RA (mas/yr)",
            "ylabel": "Proper Motion Dec (mas/yr)",
            "invert_y": False,
        },
        {
            "name": "CMD",
            "x": bp_rp,
            "y": M_G,
            "xlabel": "Gaia BP - RP",
            "ylabel": "Absolute Gaia G",
            "invert_y": True,
        },
    ]

    fig, axs = plt.subplots(3, 3, figsize=(15, 15))

    false_color = "tab:blue"
    true_color = "tab:red"

    for row, view in enumerate(views):
        xlim = (np.nanmin(view["x"]), np.nanmax(view["x"]))
        ylim = (np.nanmin(view["y"]), np.nanmax(view["y"]))

        for col, (flag_label, flag_col) in enumerate(flags.items()):
            ax = axs[row, col]

            selected = df[flag_col]
            not_selected = ~selected

            ax.scatter(
                view["x"][not_selected],
                view["y"][not_selected],
                s=5,
                color=false_color,
                label="False",
                alpha=0.7
            )

            ax.scatter(
                view["x"][selected],
                view["y"][selected],
                s=5,
                color=true_color,
                label="True",
                alpha=0.9
            )

            ax.set_xlim(xlim)
            ax.set_ylim(ylim)

            if view["invert_y"]:
                ax.invert_yaxis()

            ax.set_xlabel(view["xlabel"])
            ax.set_ylabel(view["ylabel"])
            ax.set_title(f"{view['name']}: {flag_label}")
            ax.legend(markerscale=3)

    plt.tight_layout()
    plt.show()

    return df


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_continuous_and_founding_comparison(df):
    
    #Change RA to be wrapped from -180 to 180 degrees instead of 0 to 360 for better visualization in spatial plot
    df["ra_wrapped"] = df["RA"].copy()
    mask = df["ra_wrapped"] > 180
    df.loc[mask, "ra_wrapped"] = df.loc[mask, "ra_wrapped"] - 360
    
    bp_rp = df["bp"] - df["rp"]
    M_G = df["g"] - 5 * np.log10(df["d"]) + 5

    views = [
        {
            "name": "Spatial",
            "x": df["ra_wrapped"],
            "y": df["Dec"],
            "xlabel": "Right Ascension (degrees)",
            "ylabel": "Declination (degrees)",
            "invert_y": False,
        },
        {
            "name": "Proper Motion",
            "x": df["pmra"],
            "y": df["pmdec"],
            "xlabel": "Proper Motion RA (mas/yr)",
            "ylabel": "Proper Motion Dec (mas/yr)",
            "invert_y": False,
        },
        {
            "name": "CMD",
            "x": bp_rp,
            "y": M_G,
            "xlabel": "Gaia BP - RP",
            "ylabel": "Absolute Gaia G",
            "invert_y": True,
        },
    ]

    fig, axs = plt.subplots(3, 3, figsize=(16, 15))

    for row, view in enumerate(views):
        xlim = (np.nanmin(view["x"]), np.nanmax(view["x"]))
        ylim = (np.nanmin(view["y"]), np.nanmax(view["y"]))

        # Pyoung continuous colour map
        sc0 = axs[row, 0].scatter(
            view["x"],
            view["y"],
            c=df["Pyoung"],
            s=5,
            cmap="viridis"
        )
        axs[row, 0].set_title(f"{view['name']}: Pyoung")
        plt.colorbar(sc0, ax=axs[row, 0], label="Pyoung")

        # Strength continuous colour map
        sc1 = axs[row, 1].scatter(
            view["x"],
            view["y"],
            c=df["strength"],
            s=5,
            cmap="viridis"
        )
        axs[row, 1].set_title(f"{view['name']}: Strength")
        plt.colorbar(sc1, ax=axs[row, 1], label="Strength")

        # Founding binary flag
        founding_true = df["founding"].astype(bool)
        founding_false = ~founding_true

        axs[row, 2].scatter(
            view["x"][founding_false],
            view["y"][founding_false],
            s=5,
            color="tab:blue",
            label="False",
            alpha=0.7
        )

        axs[row, 2].scatter(
            view["x"][founding_true],
            view["y"][founding_true],
            s=5,
            color="tab:red",
            label="True",
            alpha=0.9
        )

        axs[row, 2].set_title(f"{view['name']}: Founding Flag")
        axs[row, 2].legend(markerscale=3)

        for col in range(3):
            axs[row, col].set_xlim(xlim)
            axs[row, col].set_ylim(ylim)

            if view["invert_y"]:
                axs[row, col].invert_yaxis()

            axs[row, col].set_xlabel(view["xlabel"])
            axs[row, col].set_ylabel(view["ylabel"])

    plt.tight_layout()
    plt.show()


In [ ]:
df_CG26 = pd.read_csv("SCYA/CLUSTER_EXTENDED_CG26.csv", encoding="utf-8")

#### 6)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Choose whichever SCYA data set you want to investigate.
df_CG26 = pd.read_csv("SCYA/CLUSTER_EXTENDED_CG26.csv")
df_6 = pd.read_csv("SCYA/SCYA6_AllRVs.csv")
df_69 = pd.read_csv("SCYA/SCYA69_AllRVs.csv")
df_72 = pd.read_csv("SCYA/SCYA72_AllRVs.csv")

In [ ]:
def plot_strength_filtered_cmd(df, strength_cut=0.5, name = "default"):
    df = df.copy()

    strength_selected = df["strength"] > strength_cut
    founding_stars = df["founding"] == 1

    # Mutually exclusive groups
    red_points = strength_selected
    yellow_points = founding_stars & ~strength_selected
    black_points = ~(strength_selected | founding_stars)

    bp_rp = df["bp"] - df["rp"]
    M_G = df["g"] - 5 * np.log10(df["d"]) + 5

    background_size = 8
    selected_size = 18

    plt.figure(figsize=(7, 8))

    # Bottom layer: all other points
    plt.scatter(
        bp_rp[black_points],
        M_G[black_points],
        s=background_size,
        color="black",
        alpha=1.0,
        label="All other points",
    )

    # Middle layer: founding points not already strength-selected
    plt.scatter(
        bp_rp[yellow_points],
        M_G[yellow_points],
        s=selected_size,
        color="goldenrod",
        alpha=1.0,
        label="founding == 1",
    )

    # Top layer: strength-selected points
    plt.scatter(
        bp_rp[red_points],
        M_G[red_points],
        s=selected_size,
        color="red",
        alpha=1.0,
        label=f"strength > {strength_cut}",
    )

    plt.gca().invert_yaxis()
    plt.xlabel("Gaia BP - RP")
    plt.ylabel("Absolute Gaia G")
    plt.title(f"CMD with Strength Cut: strength > {strength_cut}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"figures/{name}_CMD_cut.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
# Example:
plot_strength_filtered_cmd(df_CG26, strength_cut=0.95, name="CG26")

# Try other data sets/cuts:
# plot_strength_filtered_cmd(df_6, strength_cut=0.2, name="SCYA6")
# plot_strength_filtered_cmd(df_69, strength_cut=0.2, name="SCYA69")
# plot_strength_filtered_cmd(df_72, strength_cut=0.2, name="SCYA72")

In [ ]:
def plot_strength_restricted_3x3(df, strength_cut=0.5, name="default"):
    df = df.copy()

    # Use the full dataframe to set fixed axes and color scales.
    full_df = df.copy()

    # Use the filtered dataframe only for the plotted points.
    plot_df = df[df["strength"] > strength_cut].copy()

    if plot_df.empty:
        raise ValueError("No points pass the requested strength cut.")

    def make_views(data):
        data = data.copy()

        # RA wrapping
        data["ra_wrapped"] = data["RA"].copy()
        mask = data["ra_wrapped"] > 180
        data.loc[mask, "ra_wrapped"] = data.loc[mask, "ra_wrapped"] - 360

        # CMD quantities
        bp_rp = data["bp"] - data["rp"]
        M_G = data["g"] - 5 * np.log10(data["d"]) + 5

        return [
            ["Spatial", data["ra_wrapped"], data["Dec"],
             "Right Ascension (degrees)", "Declination (degrees)", False],

            ["Proper Motion", data["pmra"], data["pmdec"],
             "Proper Motion RA (mas/yr)", "Proper Motion Dec (mas/yr)", False],

            ["CMD", bp_rp, M_G,
             "Gaia BP - RP", "Absolute Gaia G", True],
        ]

    views = make_views(plot_df)
    full_views = make_views(full_df)

    pyoung_vmin = np.nanpercentile(full_df["Pyoung"], 1)
    pyoung_vmax = np.nanpercentile(full_df["Pyoung"], 99)

    strength_vmin = np.nanpercentile(full_df["strength"], 1)
    strength_vmax = np.nanpercentile(full_df["strength"], 99)

    fig, axs = plt.subplots(3, 3, figsize=(16, 15))
    fig.suptitle(f"Stars with strength > {strength_cut}", fontsize=16, y=1.02)

    founding_true = plot_df["founding"].astype(bool)
    founding_false = ~founding_true

    for row, (view, full_view) in enumerate(zip(views, full_views)):
        name, x, y, xlabel, ylabel, invert_y = view

        x_full = full_view[1]
        y_full = full_view[2]

        xlim = (np.nanmin(x_full), np.nanmax(x_full))
        ylim = (np.nanmin(y_full), np.nanmax(y_full))

        pyoung_order = np.argsort(plot_df["Pyoung"].values)
        strength_order = np.argsort(plot_df["strength"].values)

        sc0 = axs[row, 0].scatter(
            x.iloc[pyoung_order] if hasattr(x, "iloc") else x[pyoung_order],
            y.iloc[pyoung_order] if hasattr(y, "iloc") else y[pyoung_order],
            c=plot_df["Pyoung"].iloc[pyoung_order],
            s=6,
            cmap="viridis",
            alpha=0.75,
            vmin=pyoung_vmin,
            vmax=pyoung_vmax
        )
        axs[row, 0].set_title(f"{name}: Pyoung")
        plt.colorbar(sc0, ax=axs[row, 0], label="Pyoung")

        sc1 = axs[row, 1].scatter(
            x.iloc[strength_order] if hasattr(x, "iloc") else x[strength_order],
            y.iloc[strength_order] if hasattr(y, "iloc") else y[strength_order],
            c=plot_df["strength"].iloc[strength_order],
            s=6,
            cmap="viridis",
            alpha=0.75,
            vmin=strength_vmin,
            vmax=strength_vmax
        )
        axs[row, 1].set_title(f"{name}: Strength")
        plt.colorbar(sc1, ax=axs[row, 1], label="Strength")

        axs[row, 2].scatter(
            x[founding_false],
            y[founding_false],
            s=3,
            color="tab:blue",
            label="False",
            alpha=0.25
        )

        axs[row, 2].scatter(
            x[founding_true],
            y[founding_true],
            s=35,
            color="tab:red",
            edgecolor="black",
            linewidth=0.4,
            label="True",
            alpha=1.0
        )

        axs[row, 2].set_title(f"{name}: Founding Flag")
        axs[row, 2].legend(markerscale=1.5)

        for col in range(3):
            axs[row, col].set_xlim(xlim)
            axs[row, col].set_ylim(ylim)

            if invert_y:
                axs[row, col].invert_yaxis()

            axs[row, col].set_xlabel(xlabel)
            axs[row, col].set_ylabel(ylabel)

    plt.tight_layout()
    plt.savefig(f"figures/{name}_3x3.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
plot_strength_restricted_3x3(df_CG26, strength_cut=0.2, name="CG26")

In [ ]:
plot_strength_filtered_cmd(df_6, strength_cut=0.5)

In [ ]:
plot_strength_restricted_3x3(df_6, strength_cut=0)

In [ ]:
plot_strength_restricted_3x3(df_6, strength_cut=0.16, name = "SCYA6")

In [ ]:
plot_strength_filtered_cmd(df_69, strength_cut=0.5, name = "SCYA69")

In [ ]:
plot_strength_restricted_3x3(df_69, strength_cut=0, name="SCYA69")

In [ ]:
plot_strength_restricted_3x3(df_69, strength_cut=0.2, name="SCYA69")

In [ ]:
plot_strength_filtered_cmd(df_72, strength_cut=0.4, name = "SCYA72")

In [ ]:
plot_strength_restricted_3x3(df_72, strength_cut=0, name = "SCYA72")

In [ ]:
plot_strength_restricted_3x3(df_72, strength_cut=0.2, name = "SCYA72")

## Intro Assignment P2

In [ ]:
import pandas as pd

file = "SCYA/Old_Isochrones (>1 Gyr).txt"

with open(file) as f:
    for line in f:
        if line.startswith("# Zini"):
            columns = line.replace("#", "").split()
            break

old_iso = pd.read_csv(
    file,
    comment="#",
    sep=r"\s+",
    names=columns
)

old_iso.head()

In [ ]:
file = "SCYA/Young_Isochrones (1-80Myr).txt"

with open(file) as f:
    for line in f:
        if line.startswith("# Zini"):
            columns = line.replace("#", "").split()
            break

young_iso = pd.read_csv(
    file,
    comment="#",
    sep=r"\s+",
    names=columns
)

young_iso.head()

In [ ]:
print(young_iso.columns)

In [ ]:
# Create a new column of Age in Myr from logAge
old_iso["Age"] = (10 ** old_iso["logAge"]) / 1e6
young_iso["Age"] = (10 ** young_iso["logAge"]) / 1e6

In [ ]:
df = pd.read_csv("SCYA/SCYA69_AllRVs.csv", encoding="utf-8")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def isochrone_overlay(iso, data, Age_Myr, pmem_cut=None, name=None):
    # Filter data by Pmem
    data = data.copy()
    if pmem_cut is not None:
        data = data[data["Pmem"] > pmem_cut].copy()

    # Plot low Pmem first, high Pmem on top
    data = data.sort_values("Pmem")

    # Observed CMD
    bp_rp_data = data["bp"] - data["rp"]
    M_G = data["g"] - 5 * np.log10(data["d"]) + 5

    # Copy isochrone
    iso = iso.copy()

    # Convert logAge in years to Age in Myr
    iso["Age_Myr"] = 10**iso["logAge"] / 1e6

    # Find closest available isochrone age by minimizing absolute difference and selecting the smallest one (closest)
    available_ages = np.sort(iso["Age_Myr"].unique())
    nearest_age = available_ages[np.argmin(np.abs(available_ages - Age_Myr))]

    # Select that isochrone
    iso_age = iso[np.isclose(iso["Age_Myr"], nearest_age)].copy()

    # Clean weird advanced evolutionary phases (that were making the plot wild before)
    # Label is the evolutionary stage so including all of them included them becoming very bright and red and dominating the plot
    # label 0/1 are usually the clean main-sequence-ish parts -> can go up to 7 without including craziness
    iso_age = iso_age[iso_age["label"] <= 7].copy()

    # Sort by initial mass so the line connects in physical order
    iso_age = iso_age.sort_values("Mini")

    # Isochrone CMD
    bp_rp_iso = iso_age["G_BPmag"] - iso_age["G_RPmag"]
    G_iso = iso_age["Gmag"]

    # Plot
    plt.figure(figsize=(6, 7))

    sc = plt.scatter(
        bp_rp_data,
        M_G,
        c=data["Pmem"],
        s=8,
        cmap="viridis",
        alpha=0.8,
        label="Data"
    )

    plt.plot(
        bp_rp_iso,
        G_iso,
        color="red",
        linewidth=2,
        label=f"Isochrone = {nearest_age:.2f} Myr"
    )

    plt.colorbar(sc, label="Pmem")
    plt.xlabel("Gaia BP - RP")
    plt.ylabel("Absolute Gaia G")
    plt.title(f"CMD with {nearest_age:.2f} Myr Isochrone")
    plt.gca().invert_yaxis()
    plt.grid()
    plt.legend()
    plt.tight_layout()

    if name is not None:
        plt.savefig(f"figures/{name}_isochrone_fit.png", dpi=300, bbox_inches="tight")

    plt.show()

    

In [ ]:
isochrone_overlay(young_iso, df, Age_Myr=20)

In [ ]:
isochrone_overlay(old_iso, df, Age_Myr=2500)

In [ ]:
df_6 = pd.read_csv("SCYA/SCYA6_AllRVs.csv", encoding="utf-8")

In [ ]:
isochrone_overlay(young_iso, df_6, Age_Myr=5)

In [ ]:
isochrone_overlay(old_iso, df_6, Age_Myr=100000)

In [ ]:
df_72 = pd.read_csv("SCYA/SCYA72_AllRVs.csv", encoding="utf-8")

In [ ]:
isochrone_overlay(young_iso, df_72, Age_Myr=24)

In [ ]:
isochrone_overlay(old_iso, df_72, Age_Myr=1000)

In [ ]:
df_CG26 = pd.read_csv("SCYA/CLUSTER_EXTENDED_CG26.csv", encoding="utf-8")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def isochrone_overlay_plain(iso, data, Age_Myr):
    data = data.copy()

    # Observed CMD
    bp_rp_data = data["bp"] - data["rp"]
    M_G = data["g"] - 5 * np.log10(data["d"]) + 5

    # Isochrone
    iso = iso.copy()
    iso["Age_Myr"] = 10**iso["logAge"] / 1e6

    available_ages = np.sort(iso["Age_Myr"].unique())
    nearest_age = available_ages[np.argmin(np.abs(available_ages - Age_Myr))]

    iso_age = iso[np.isclose(iso["Age_Myr"], nearest_age)].copy()
    iso_age = iso_age[iso_age["label"] <= 7].copy()
    iso_age = iso_age.sort_values("Mini")

    bp_rp_iso = iso_age["G_BPmag"] - iso_age["G_RPmag"]
    G_iso = iso_age["Gmag"]

    # Plot
    plt.figure(figsize=(6, 7))

    plt.scatter(
        bp_rp_data,
        M_G,
        s=8,
        alpha=0.6,
        label="Data"
    )

    plt.plot(
        bp_rp_iso,
        G_iso,
        color="red",
        linewidth=2,
        label=f"Isochrone = {nearest_age:.2f} Myr"
    )

    plt.xlabel("Gaia BP - RP")
    plt.ylabel("Absolute Gaia G")
    plt.title(f"CMD with {nearest_age:.2f} Myr Isochrone")
    plt.gca().invert_yaxis()
    plt.grid()
    plt.legend()
    plt.show()

In [ ]:
isochrone_overlay_plain(young_iso, df_CG26, Age_Myr=4)

In [ ]:
isochrone_overlay_plain(young_iso, df_CG26, Age_Myr=1000)

In [ ]:
import pandas as pd
from pyvo.dal import TAPService

def load_83_cluster_isochrones():
    tap = TAPService("https://tapvizier.cds.unistra.fr/TAPVizieR/tap")
    table_name = "J/A+A/690/A16/tableb1"

    result = tap.search(f'SELECT * FROM "{table_name}"')
    iso = result.to_table().to_pandas()

    return iso


In [ ]:
iso = load_83_cluster_isochrones()
iso.head()

In [ ]:
iso["Cluster"].unique()

In [ ]:
iso.to_csv("rottensteiner_83_cluster_isochrones.csv", index=False)
# Saving iso to csv so I don't need to re-download every time

In [ ]:
iso = pd.read_csv("rottensteiner_83_cluster_isochrones.csv")

In [ ]:
iso["Cluster"].unique()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def vizier_cluster_isochrone_overlay(
    iso,
    data,
    cluster_name,
    catalog="DR3",
    col_filt="BP-RP",
    absmag_filt="G",
    pmem_cut=None
):
    data = data.copy()
    iso = iso.copy()

    # Optional: only cut if you actually pass pmem_cut
    # If pmem_cut=None, this keeps ALL data.
    if pmem_cut is not None:
        data = data[data["Pmem"] > pmem_cut].copy()

    # Important: sort BEFORE calculating x, y, and colors
    # This keeps Pmem colors matched to the right points.
    if "Pmem" in data.columns:
        data = data.sort_values("Pmem")

    # Observed CMD from your dataset
    bp_rp_data = data["bp"] - data["rp"]
    M_G = data["g"] - 5 * np.log10(data["d"]) + 5

    # Pull one empirical isochrone from the full VizieR iso dataframe
    cluster_iso = iso[
        (iso["Cluster"] == cluster_name) &
        (iso["Catalog"] == catalog) &
        (iso["ColFilt"] == col_filt) &
        (iso["AbsmagFilt"] == absmag_filt)
    ].copy()

    if len(cluster_iso) == 0:
        print("No matching isochrone found.")
        print("Available options:")
        print(
            iso[["Cluster", "Catalog", "ColFilt", "AbsmagFilt", "logAge"]]
            .drop_duplicates()
            .head(50)
        )
        return

    cluster_iso = cluster_iso.sort_values("isox")

    age_myr = 10**cluster_iso["logAge"].iloc[0] / 1e6

    plt.figure(figsize=(6, 7))

    if "Pmem" in data.columns:
        sc = plt.scatter(
            bp_rp_data,
            M_G,
            c=data["Pmem"],
            s=8,
            cmap="viridis",
            alpha=0.8,
            label="Data"
        )
        plt.colorbar(sc, label="Pmem")
    else:
        plt.scatter(
            bp_rp_data,
            M_G,
            s=8,
            alpha=0.8,
            label="Data"
        )

    plt.plot(
        cluster_iso["isox"],
        cluster_iso["isoy"],
        color="red",
        linewidth=2,
        label=f"{cluster_name} Isochrone = {age_myr:.2f} Myr"
    )

    plt.xlabel("Gaia BP - RP")
    plt.ylabel("Absolute Gaia G")
    plt.title(f"CMD with {cluster_name} Empirical Isochrone")
    plt.gca().invert_yaxis()
    plt.grid()
    plt.legend()
    plt.show()

In [ ]:
vizier_cluster_isochrone_overlay(
    iso=iso,
    data=df,
    cluster_name="ASCC_127"
)
